## Quick Start

In [1]:
from sft_wick import Field, Vertex, Action, compute_moment

# Define scalar fields
phi = Field('phi', 'physical')
psi = Field('psi', 'response')

# Compute <psi(x) phi(x) phi(x) phi(x)>_{S_0}
obs = [psi('x'), phi('x'), phi('x'), phi('x')]
result = compute_moment(obs, Action(vertices=[]), order=0)
print(result.order(0).to_latex())
# Output: 3 R(x, x) C(x, x)

3 R(x, x) C(x, x)


## 

In [2]:
from sft_wick import Field

# # Scalar fields (single component)
# phi = Field('phi', 'physical')
# psi = Field('psi', 'response')

# # Scalar: phi(spatial_arg)
# op = phi('x')          # φ(x)

# Multi-component fields
phi = Field('phi', 'physical', n_components=3)
psi = Field('psi', 'response', n_components=3)

# Multi-component: phi(component_index, spatial_arg)
op = phi('a', 'x')     # φ_a(x)
op = psi('b', 'y')     # ψ_b(y)


from sft_wick import Vertex

# Local vertex: ∫ F_{ijk} φ_i(x) φ_j(x) ψ_k(x) dx
# All fields share the same spatial argument.
v1 = Vertex(fields=[phi, phi, psi], coupling='F')

# Non-local vertex: ∬ K_{ij}(x, x') ψ_i(x) ψ_j(x') dx dx'
# Each field gets its own spatial argument.
v2 = Vertex(fields=[psi, psi, psi], coupling='K', local=False)

In [3]:
from sft_wick import Action, compute_moment

action = Action(vertices=[v1])
obs = [psi('a', 'x'), phi('b', 'x'), phi('c', 'x'), phi('d', 'x')]

result = compute_moment(obs, action, order=3)

# Access individual orders
print(result.order(0).to_latex())
print(result.order(1).to_latex())
print(result.order(2).to_latex())
print(result.order(3).to_latex())


# Full result
print(result.to_latex())

R_{ba}(x, x) C_{cd}(x, x) + R_{ca}(x, x) C_{bd}(x, x) + R_{da}(x, x) C_{bc}(x, x)
0
\sum_{i_5=1}^{3} \sum_{i_4=1}^{3} \sum_{i_3=1}^{3} \sum_{i_2=1}^{3} \sum_{i_1=1}^{3} \sum_{i_0=1}^{3} \int \mathrm{d}y_1\, \int \mathrm{d}y_0\, \frac{1}{2} F_{i_0i_1i_2} F_{i_3i_4i_5} \left(R_{ba}(x, x) R_{ci_2}(x, y_0) R_{di_5}(x, y_1) C_{i_0i_1}(y_0, y_0) C_{i_3i_4}(y_1, y_1) + R_{ba}(x, x) R_{ci_2}(x, y_0) R_{di_5}(x, y_1) C_{i_0i_3}(y_0, y_1) C_{i_1i_4}(y_0, y_1) + R_{ba}(x, x) R_{ci_2}(x, y_0) R_{di_5}(x, y_1) C_{i_0i_4}(y_0, y_1) C_{i_1i_3}(y_0, y_1) + R_{ba}(x, x) R_{di_2}(x, y_0) R_{ci_5}(x, y_1) C_{i_0i_1}(y_0, y_0) C_{i_3i_4}(y_1, y_1) + R_{ba}(x, x) R_{di_2}(x, y_0) R_{ci_5}(x, y_1) C_{i_0i_3}(y_0, y_1) C_{i_1i_4}(y_0, y_1) + R_{ba}(x, x) R_{di_2}(x, y_0) R_{ci_5}(x, y_1) C_{i_0i_4}(y_0, y_1) C_{i_1i_3}(y_0, y_1) + R_{ca}(x, x) R_{bi_2}(x, y_0) R_{di_5}(x, y_1) C_{i_0i_1}(y_0, y_0) C_{i_3i_4}(y_1, y_1) + R_{ca}(x, x) R_{bi_2}(x, y_0) R_{di_5}(x, y_1) C_{i_0i_3}(y_0, y_1) C_{i_1i_4}(y_0, y_1) 

In [4]:
# Draw all diagrams
result.draw_diagrams()

# Draw only diagrams at a specific order
result.draw_diagrams(order=1)

# Access diagram topology
for d_info in result.diagrams_by_order[1]:
    fd = d_info.to_feynman_diagram()
    print(fd.summary())
    print(f"  Loops: {fd.n_loops}, Connected: {fd.is_connected}")

No diagrams to draw.


In [5]:
from sft_wick import LaTeXFormatter

# Default names
print(result.order(0).to_latex())
# C_{ab}(x, y)

# Custom propagator names
fmt = LaTeXFormatter(propagator_names={
    'C': 'G',
    'R': r'R^{\mathrm{ret}}'
})
print(fmt.format(result.order(0)))
# G_{ab}(x, y)

# LaTeX align environment for order-by-order display
print(fmt.format_aligned(result.order_terms))

R_{ba}(x, x) C_{cd}(x, x) + R_{ca}(x, x) C_{bd}(x, x) + R_{da}(x, x) C_{bc}(x, x)
R^{\mathrm{ret}}_{ba}(x, x) G_{cd}(x, x) + R^{\mathrm{ret}}_{ca}(x, x) G_{bd}(x, x) + R^{\mathrm{ret}}_{da}(x, x) G_{bc}(x, x)
\begin{align}
  O(0) &= R^{\mathrm{ret}}_{ba}(x, x) G_{cd}(x, x) + R^{\mathrm{ret}}_{ca}(x, x) G_{bd}(x, x) + R^{\mathrm{ret}}_{da}(x, x) G_{bc}(x, x) \\
  O(1) &= 0 \\
  O(2) &= \sum_{i_5=1}^{3} \sum_{i_4=1}^{3} \sum_{i_3=1}^{3} \sum_{i_2=1}^{3} \sum_{i_1=1}^{3} \sum_{i_0=1}^{3} \int \mathrm{d}y_1\, \int \mathrm{d}y_0\, \frac{1}{2} F_{i_0i_1i_2} F_{i_3i_4i_5} \left(R^{\mathrm{ret}}_{ba}(x, x) R^{\mathrm{ret}}_{ci_2}(x, y_0) R^{\mathrm{ret}}_{di_5}(x, y_1) G_{i_0i_1}(y_0, y_0) G_{i_3i_4}(y_1, y_1) + R^{\mathrm{ret}}_{ba}(x, x) R^{\mathrm{ret}}_{ci_2}(x, y_0) R^{\mathrm{ret}}_{di_5}(x, y_1) G_{i_0i_3}(y_0, y_1) G_{i_1i_4}(y_0, y_1) + R^{\mathrm{ret}}_{ba}(x, x) R^{\mathrm{ret}}_{ci_2}(x, y_0) R^{\mathrm{ret}}_{di_5}(x, y_1) G_{i_0i_4}(y_0, y_1) G_{i_1i_3}(y_0, y_1) + R^{\mathrm{ret